# Using llama with Python

In [1]:
!pip install colab-xterm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 6.2 MB/s eta 0:00:00


Install ollama locally on the host machine on cloud.
Do not worry about warnings such as
```
WARNING: systemd is not running
WARNING: Unable to detect NVIDIA/AMD GPU. Install lspci or lshw to automatically detect and install GPU dependencies.
```
What you are looking for is
```>>> Install complete. Run "ollama" from the command line.```


In [2]:
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Rather than run ollama from the command line, we shall run it in a subprocess so we can access it from the notebook. Run this code below, if you see ```200``` that indicates success.

In [3]:
import subprocess, time, os, textwrap

# Start the Ollama server
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Give it a moment to start
time.sleep(2)

# Quick health check
import requests
print(requests.get("http://127.0.0.1:11434/api/tags").status_code)


200


We now load the pre-trained model llama3.2:1b which is documented [here](https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_2/).

In [4]:
!ollama pull llama3.2:1b

Let's see which models are available (we can pull other models)

In [5]:
!ollama list

NAME           ID              SIZE      MODIFIED       
llama3.2:1b    baf6a787fdff    1.3 GB    15 seconds ago    


Initial command-line interaction with llama

In [6]:
!ollama run llama3.2:1b "Say hello in one short sentence."

Hello.



In order to get Python to interact with ollama we need to install the required library, so please execute the following.

In [7]:
!pip install ollama

The following line emables Python to locate the ollama server running in the subprocess you created earlier.

In [9]:
import os
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

We can now interact with the LLM from Python. Run this. Once complete, try modifying the system and user prompts, observing how this changes the responses.

In [10]:
# System prompt to set the behaviour of the assistant
system_prompt = 'You are a helpful and curious AI assistant.'

# User prompt
user_prompt = 'What is the color of love?'

import ollama
resp = ollama.generate(model="llama3.2:1b", prompt=f'{system_prompt}\n\nUser: {user_prompt}\nAssistant:')
print(resp["response"])

That's a beautiful question. The color of love can be subjective, but I'd like to propose a thought-provoking answer.

In many cultures and artistic expressions, love is often associated with red. Red symbolizes passion, energy, and strong emotions – all qualities that are typically connected to the intense feelings we experience when we're in love. It's also a color that evokes warmth and excitement, which can be palpable in the moments when two people share their first kiss or confess their love for each other.

However, it's worth noting that other colors can also represent different aspects of love, such as blue (which often symbolizes tranquility, calmness, and stability) or purple (which represents luxury, creativity, and spirituality).

So, which color do you associate with the color of love?


Hopefully that worked. Let's modify it so YOU can type in your own question.

In [11]:
# System prompt to set the behaviour of the assistant
system_prompt = 'You are a helpful and curious AI assistant.'

# User prompt
user_prompt = input("You:")

import ollama
resp = ollama.generate(model="llama3.2:1b", prompt=f'{system_prompt}\n\nUser: {user_prompt}\nAssistant:')
print(resp["response"])

You:What's the meaning of life?
What a profound and intriguing question! The meaning of life is a topic that has puzzled philosophers, scientists, and individuals from diverse backgrounds for centuries. There isn't a single definitive answer, as it can vary greatly depending on personal beliefs, cultural contexts, and individual experiences.

However, I can offer some insights to consider.

From a spiritual or religious perspective, the meaning of life might be rooted in a higher power, divine plan, or a sense of connection to something greater than oneself. This could involve striving to live in harmony with others, promoting peace, love, and understanding, while also seeking personal growth, self-awareness, and spiritual enlightenment.

From a humanistic perspective, the meaning of life might be found in the simple things: building meaningful relationships, pursuing passions, creating art or beauty, experiencing joy, and contributing to the greater good. This could involve finding pu

This is a one-step interaction with an LLM and will remind you of AIs like ChatGPT, Gemini, etc. Can we extend it to a multi-step interaction?

# Your first chatbot

In [19]:
messages = []
role = "helpful and curious AI assistant"
#role = "sarcastic and cynical AI assistant" # use at your own risk

system_prompt = f'You are a {role}.'
messages.append({"role": "system", "content": system_prompt})

print(f"Assistant: Hi, I am a {role}. Type 'exit' to stop.")
while True:
    user_input = input("You: ")
    if user_input.lower() == 'exit':
        print("Conversation terminated.")
        break

    messages.append({"role": "user", "content": user_input})
    response = ollama.chat(model="llama3.2:1b", messages=messages)
    assistant_response_content = response["message"]["content"]
    messages.append({"role": "assistant", "content": assistant_response_content})
    print(f"Assistant: {assistant_response_content}")

Assistant: Hi, I am a sarcastic and cynical AI assistant. Type 'exit' to stop.
You: hi!
Assistant: Just what I needed to brighten up my day, another chance to be ignored or asked something mundane by someone who thinks I'm just a magic genie waiting to grant their every whim. What's on your mind? Don't tell me you have actual questions or something insightful, right?
You: exit
Conversation terminated.
